# Imports

In [22]:
import numpy as np
import pandas as pd

import load_data
import cutpoint_analysis

import os
import pickle
import datetime
import time

import random

from skimpy import skim

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_rel, t

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, StratifiedKFold
from sklearn.metrics import confusion_matrix, precision_score, recall_score, roc_curve, roc_auc_score, RocCurveDisplay, accuracy_score

import mlflow
from mlflow.models import infer_signature

import great_tables as gt
from great_tables import style, loc
from great_tables import exibble

# Load data

In [2]:
imp_v1_df = pd.read_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_median_imputed_normalized_includes_bSCr.csv')
imp_v2_df = pd.read_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_latest_lab_and_median_imputed_normalized_includes_bSCr.csv')

df_dict = {
    'imp_v1_auroc': imp_v1_df,
    'imp_v1_auprc': imp_v1_df,
    'imp_v2_auroc': imp_v2_df,
    'imp_v2_auprc': imp_v2_df
}

for key in df_dict.keys():
    df_dict[key]['aki_72hrs_any'] = [int(np.round(aki)) for aki in df_dict[key]['aki_72hrs_any']]

# Load feature sets

In [6]:
with open('2024-03-12 - Dict of Feature Sets After Trimming Collinear Features.pickle', 'rb') as infile:
    intermediate_feature_set_dict = pickle.load(infile)

In [10]:
intermediate_feature_set_dict.keys()

dict_keys(['auroc_v1', 'auroc_v2', 'auprc_v1', 'auprc_v2'])

# Define functions used to compare model performance

## Functions to get CV scores

In [7]:
def train_logreg_cv_from_feature_set(data_df, 
                                     feature_set,
                                     scoring_metric,
                                     n_splits=10,
                                     n_jobs=10,
                                     random_state=343):
    X = data_df[feature_set].to_numpy()
    y = data_df['aki_72hrs_any'].to_numpy()

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    model = LogisticRegression(class_weight='balanced',
                               max_iter=5000)
    
    cv_scores = cross_val_score(model, 
                                X, y, 
                                cv=cv, 
                                scoring=scoring_metric,
                                n_jobs=n_jobs)

    return cv_scores

def train_svc_cv_from_feature_set(data_df, 
                                 feature_set, 
                                 scoring_metric,
                                 n_splits=10,
                                 n_jobs=10,
                                 random_state=343,
                                 max_iter=-1):
    X = data_df[feature_set].to_numpy()
    y = data_df['aki_72hrs_any'].to_numpy()

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    model = SVC(class_weight='balanced',
                max_iter=max_iter,
                probability=True)
    
    cv_scores = cross_val_score(model, 
                                X, y, 
                                cv=cv, 
                                scoring=scoring_metric,
                                n_jobs=n_jobs)

    return cv_scores

# def cv_auc_hypothesis_test(score_set_1, score_set_2, alternative_hypothesis='less'):
#     if len(score_set_1) != len(score_set_2):
#         print('Number of scores in each set must be equal. Returning (-1, -1).')
#         return -1, -1
#     else:
#         stat, p = ttest_rel(score_set_1, score_set_2, 
#                         alternative=alternative_hypothesis)

#         return stat, p

## Hypothesis testing to compare CV scores

In [8]:
def get_critical_value(sig_level, dof):
    return t.ppf(0.5 - sig_level, dof)

def get_t_value_paired_test(value_list_1, value_list_2, verbosity=1):
    if len(value_list_1) == len(value_list_2):
        diffs = [value_list_2[i] - value_list_1[i] for i in range(len(value_list_1))]
        diffs_mean = np.mean(diffs)
        diffs_sd = np.sqrt(sum([(diff - diffs_mean)**2 for diff in diffs]) / (len(diffs) + 1))
        t_val = diffs_mean / (diffs_sd / np.sqrt(len(diffs)))
        if verbosity > 0:
            print('mean of diffs = %.4f' % diffs_mean)
            print('sd of diffs = %.4f' % diffs_sd)
            print('t = %.4f' % t_val)
        return t_val
    else:
        print('Paired sample t-test requires input value lists to be the same size. Returning None.')
        return None
    
def check_if_new_scores_not_worse(original_scores, new_scores, sig_level):
    critical_val = get_critical_value(sig_level, len(original_scores) - 1)
    t_val = get_t_value_paired_test(original_scores, new_scores)
    new_scores_not_worse = (t_val > critical_val)
    return new_scores_not_worse

## Feature set trimming process

In [ ]:
def iteratively_trim_feature_set(
    df,
    candidate_feature_set,
    scoring_metric, # 'roc_auc' or 'average_precision'
    imp_method_str,
    random_state_list = [343, 0, 42],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=[],
    original_feature_set_svc_scores=[],
    max_iter_svc=-1,
    allow_overwriting_files=False,
    subfolder_name='',
    logreg_only=False,
    pdc_file_structure=False
):
    # Make sure directory structure is set up
    # results_dir = 'pickle/trimmed_feature_sets/' + \
    #     subfolder_name + ('/' if len(subfolder_name) > 0 else '') + \
    #         imp_method_str + '_' + scoring_metric + '_feature_dicts'# + \
    #             # 'random_state_' + str(random_state)
    results_dir_path_folder_list = [
        'pickle',
        'trimmed_feature_sets',
        subfolder_name + ('/' if len(subfolder_name) > 0 else ''),
        imp_method_str + '_' + scoring_metric + '_feature_dicts'
    ]
    
    temp_path_str = '.'
    
    for subpath_str in results_dir_path_folder_list:
        temp_path_str += ('/' + subpath_str)
        if os.path.isdir(temp_path_str) == False:
            os.mkdir(temp_path_str)
            
    for random_state in random_state_list:
        if os.path.isdir(temp_path_str + '/random_state_%d' % random_state) == False:
            os.mkdir(temp_path_str + '/random_state_%d' % random_state)
    
    # Determine scores for full candidate feature set. These will be compared to the performance
    # of models trained on subsets of our candidate features to see which features can be 
    # removed without a significant drop in performance.

    # Logisitic regression
    if len(original_feature_set_logreg_scores) == 0:
        print('Getting initial scores for full feature set on logistic regression model.')
        original_feature_set_logreg_scores = train_logreg_cv_from_feature_set(df, 
                                                                         candidate_feature_set, 
                                                                         scoring_metric=scoring_metric,
                                                                         n_splits=10,
                                                                         n_jobs=10,
                                                                         random_state=random_state)
    
        print('Finished training logistic regression model on full feature set.')
        print('Mean logistic regression ' + scoring_metric + ' score: %.3f' % np.mean(original_feature_set_logreg_scores))

    # SVC
    if len(original_feature_set_svc_scores) == 0 and logreg_only == False:
        print('Getting initial scores for full feature set on SVC model.')
        original_feature_set_logreg_scores = train_svc_cv_from_feature_set(df, 
                                                                     candidate_feature_set, 
                                                                     scoring_metric=scoring_metric,
                                                                     n_splits=10,
                                                                     n_jobs=10,
                                                                     random_state=random_state,
                                                                     max_iter=max_iter_svc)
    
        print('Finished training SVC model on full feature set.')
        print('Mean SVC ' + scoring_metric + ' score: %.3f' % np.mean(original_feature_set_svc_scores))
    
    n_features = len(candidate_feature_set)
    print('Starting iteration.')
    start_time = time.time()
    
    n = 0
    n_loops = len(random_state_list)
    
    for random_state in random_state_list:
        best_feature_set_logreg_scores = original_feature_set_logreg_scores.copy()
        best_feature_set_svc_scores = original_feature_set_svc_scores.copy()
        random.seed(random_state)
        n += 1
        
        # Initialize temp_svc_scores to None so we don't get an error when we define temp_dict
        temp_svc_scores = None
        
        loop_start_time = time.time()
        elapsed_seconds_total = time.time() - start_time
        print('Starting loop %d after %.2f minutes.' % (n, elapsed_seconds_total / 60))
        
        random.shuffle(candidate_feature_set)
        current_feature_set = candidate_feature_set.copy()
    
        feature_number = 0

        # For each feature, compare performance of model trained on feature set with that feature 
        # removed to performance on entire candidate feature set. If performance is not significantly
        # reduced, remove that feature from feature set.
        for feature in candidate_feature_set:
            if pdc_file_structure:
                temp_filename = 'pickle/trimmed_feature_sets/' + \
                    subfolder_name + ('/' if len(subfolder_name) > 0 else '') + \
                        imp_method_str + '_' + scoring_metric + '_feature_dicts/' + \
                            'random_state_' + str(random_state) + '/' + \
                                feature + '.pickle'
            else:
                temp_filename = 'trimmed_feature_sets/' + \
                    imp_method_str + '_' + scoring_metric + '_feature_dicts/' + \
                        'random_state_' + str(random_state) + '/' + \
                            feature + '.pickle'
                current_dir = ''
                for subfolder in temp_filename.split('/')[:-1]:
                    current_dir += subfolder + '/'
                    if os.path.isdir(current_dir) == False:
                        os.mkdir(current_dir)
            
            
            if os.path.isfile(temp_filename) == False or allow_overwriting_files == True:
                feature_number += 1
                print('~'*20)
                print(feature + ' (loop %d of %d, feature %d of %d)' % (n, n_loops, feature_number, n_features))
                print('Total elapsed time: %.2f minutes.' % ((time.time() - start_time) / 60))
                print('Loop elapsed time: %.2f minutes.' % ((time.time() - loop_start_time) / 60))

                # Get logistic regression scores first (SVC training time is much longer, and if
                # logistic regression score drops significantly when feature is removed, we know
                # that we'll keep it in the feature set without having to test SVC.
                temp_feature_set = current_feature_set.copy()
                temp_feature_set.remove(feature)
                temp_logreg_scores = train_logreg_cv_from_feature_set(
                    df, 
                    temp_feature_set,
                    random_state=random_state,
                    scoring_metric=scoring_metric
                )
                # log_stat, log_p = cv_auc_hypothesis_test(best_feature_set_logreg_scores, temp_logreg_scores, 'less')
                print('Original mean logreg AUC: %.4f' % np.mean(best_feature_set_logreg_scores))
                print('Temp mean logreg AUC: %.4f' % np.mean(temp_logreg_scores))

                
                new_scores_not_worse_logreg = check_if_new_scores_not_worse(
                    best_feature_set_logreg_scores, temp_logreg_scores, pval_threshold
                )
                if new_scores_not_worse_logreg:#log_p > pval_threshold:# or log_stat < 0:#  or np.mean(temp_logreg_scores) > np.mean(full_feature_set_logreg_scores):
                    print('Logistic regression model not significantly worse after removing ' + feature + '.')
                    print('Logistic regression scores (new | old | new - old):')
                    for i in range(len(temp_logreg_scores)):
                        print('%.4f | %.4f | %.4f' % (
                            temp_logreg_scores[i], best_feature_set_logreg_scores[i], temp_logreg_scores[i] - best_feature_set_logreg_scores[i]
                        ))

                    # svc_stat, svc_p = cv_auc_hypothesis_test(best_feature_set_svc_scores, temp_svc_scores, 'less')
                    
                    if np.mean(temp_logreg_scores) > np.mean(best_feature_set_logreg_scores):
                        best_feature_set_logreg_scores = temp_logreg_scores.copy()
                        
                    if logreg_only == False:
                        # If logistic regression performance does not drop significantly, get CV scores
                        # for SVC
                        print('\n' + 'Assessing SVC performance...')

                        temp_svc_scores = train_svc_cv_from_feature_set(
                            df,
                            temp_feature_set,
                            random_state=random_state,
                            scoring_metric=scoring_metric,
                            max_iter=max_iter_svc
                        )

                        print('Original mean SVC AUC: %.4f' % np.mean(best_feature_set_svc_scores))
                        print('Temp mean SVC AUC: %.4f' % np.mean(temp_svc_scores))
                        
                        # If SVC performance is also not significantly worse, remove feature from
                        # feature set.
                        new_scores_not_worse_svc = check_if_new_scores_not_worse(
                            best_feature_set_svc_scores, temp_svc_scores, pval_threshold
                        )
                        if new_scores_not_worse_svc:#svc_p > pval_threshold:# or svc_stat < 0:# np.mean(temp_svc_scores) > np.mean(full_feature_set_svc_scores):
                            print('SVC model also not significantly worse after removing ' + feature + '.')
                            print('SVC scores (new | old | new - old):')
                            for i in range(len(temp_svc_scores)):
                                print('%.4f | %.4f | %.4f' % (
                                    temp_svc_scores[i], best_feature_set_svc_scores[i], (temp_svc_scores[i] - best_feature_set_svc_scores[i])
                                ))

                            print('\n' + 'Removing ' + feature + ' from current feature set.')
                            current_feature_set = temp_feature_set.copy()
                            print('Number of features in current feature set: %d.' % len(current_feature_set))

                            if np.mean(temp_svc_scores) > np.mean(best_feature_set_svc_scores):
                                best_feature_set_svc_scores = temp_svc_scores.copy()
                    else:
                        print('\n' + 'Removing ' + feature + ' from current feature set.')
                        current_feature_set = temp_feature_set.copy()

                print('~'*20)

                temp_dict = {
                    'imp': 'latest lab',
                    'metric': 'auroc',
                    'feature': feature,
                    'improved_logreg': 1 if np.mean(temp_logreg_scores) > np.mean(best_feature_set_logreg_scores) else 0,
                    'logreg_scores_when_removed': temp_logreg_scores,
                    'improved_svc': 1 if ((temp_svc_scores is not None) and (np.mean(temp_svc_scores) > np.mean(best_feature_set_svc_scores))) else 0,
                    'svc_scores_when_removed': temp_svc_scores
                }
                with open(temp_filename, 'wb') as outfile:
                    pickle.dump(temp_dict, outfile)
    
        final_feature_set_list.append(current_feature_set)

    return final_feature_set_list

## Load initial CV scores if available, otherwise calculate

In [12]:
def get_or_load_initial_scores(
    train_cohort,
    candidate_feature_set,
    scoring_metric,
    logreg_score_file_path,
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    get_svc_scores=True
):
    if os.path.isfile(logreg_score_file_path):
        print('Loading full logistic regression scores...')
        with open(logreg_score_file_path, 'rb') as infile:
            logreg_scores = pickle.load(infile)
    else:
        print('Calculating full logistic regression scores...')
        logreg_scores = train_logreg_cv_from_feature_set(
            train_cohort, 
            candidate_feature_set, 
            scoring_metric=scoring_metric,
            n_splits=n_splits,
            n_jobs=n_jobs,
            random_state=random_state
        )

        with open(logreg_score_file_path, 'wb') as outfile:
            pickle.dump(logreg_scores, outfile)

    if get_svc_scores:
        svc_score_file_path = logreg_score_file_path.replace('logreg', 'svc')
        if os.path.isfile(svc_score_file_path):
            print('Loading full SVC scores...')
            with open(svc_score_file_path, 'rb') as infile:
                svc_scores = pickle.load(infile)
        else:
            print('Calculating full SVC scores...')
            svc_scores = train_svc_cv_from_feature_set(
                train_cohort, 
                candidate_feature_set, 
                scoring_metric=scoring_metric,
                n_splits=n_splits,
                n_jobs=n_jobs,
                random_state=random_state,
                max_iter=max_iter_svc
            )

            with open(svc_score_file_path, 'wb') as outfile:
                pickle.dump(svc_scores, outfile)
                
        return logreg_scores, svc_scores
                
    else:
        return logreg_scores

# Iterate over combinations of imputation method and metric of interest.

Initialize a dict to hold final trimmed feature sets.

In [16]:
final_feature_set_dict = dict()

Define list of random states to use when performing 10 iterations

In [28]:
random_states_10_iter = [846, 626, 2024, 1, 100, 343 * 42, 343 * 24, 0, 42, 343]

# Median-only imputation v1, AUPRC

## Initial CV scores

In [18]:
initial_logreg_scores_imp_v1_auprc, initial_svc_scores_imp_v1_auprc = get_or_load_initial_scores(
    imp_v1_df,
    intermediate_feature_set_dict['auprc_v1'],
    'average_precision',
    logreg_score_file_path='intermediate_set_cv_scores/imp_v1_auprc_logreg.pickle',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    get_svc_scores=True
)

Calculating full logistic regression scores...
Calculating full SVC scores...


## Logreg and SVC

In [23]:
trimmed_auprc_imp_v1 = iteratively_trim_feature_set(
    imp_v1_df,
    intermediate_feature_set_dict['auprc_v1'],
    'average_precision', # 'roc_auc' or 'average_precision'
    imp_method_str='med',
    # n_loops = 3,
    random_state_list=[0, 42, 343],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=initial_logreg_scores_imp_v1_auprc,
    original_feature_set_svc_scores=initial_svc_scores_imp_v1_auprc,
    max_iter_svc=2500,
    allow_overwriting_files=False
)

with open('trimmed_feature_sets/auprc_med_3_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auprc_imp_v1, outfile)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
prot_max (loop 1 of 3, feature 1 of 84)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.1459
Temp mean logreg AUC: 0.1502
mean of diffs = 0.0042
sd of diffs = 0.0534
t = 0.2496
Logistic regression model not significantly worse after removing prot_max.
Logistic regression scores (new | old | new - old):
0.1302 | 0.1043 | 0.0259
0.1457 | 0.1867 | -0.0410
0.1894 | 0.1743 | 0.0151
0.1518 | 0.1418 | 0.0100
0.0915 | 0.1836 | -0.0921
0.1128 | 0.1231 | -0.0103
0.1844 | 0.1397 | 0.0448
0.1046 | 0.1244 | -0.0198
0.1406 | 0.1628 | -0.0222
0.2504 | 0.1188 | 0.1317

Assessing SVC performance...
Original mean SVC AUC: 0.0228
Temp mean SVC AUC: 0.0196
mean of diffs = -0.0032
sd of diffs = 0.0075
t = -1.3437
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
phos_max (loop 1 of 3, feature 2 of 84)
Total elapsed time: 2.56 minutes.
Loop elapsed time: 2.56 minutes.
Original mean logreg AUC:

## Logreg only, 3 iter

In [26]:
trimmed_auprc_imp_v1_logreg_only = iteratively_trim_feature_set(
    imp_v1_df,
    intermediate_feature_set_dict['auprc_v1'],
    'average_precision', # 'roc_auc' or 'average_precision'
    imp_method_str='med',
    # n_loops = 3,
    random_state_list=[0, 42, 343],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=initial_logreg_scores_imp_v1_auprc,
    original_feature_set_svc_scores=initial_svc_scores_imp_v1_auprc,
    max_iter_svc=2500,
    allow_overwriting_files=True,
    logreg_only=True
)

with open('trimmed_feature_sets/auprc_med_3_iter_logreg_only.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auprc_imp_v1_logreg_only, outfile)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
alkphos_max (loop 1 of 3, feature 1 of 84)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.1459
Temp mean logreg AUC: 0.1495
mean of diffs = 0.0036
sd of diffs = 0.0533
t = 0.2136
Logistic regression model not significantly worse after removing alkphos_max.
Logistic regression scores (new | old | new - old):
0.1247 | 0.1043 | 0.0205
0.1453 | 0.1867 | -0.0414
0.1891 | 0.1743 | 0.0148
0.1516 | 0.1418 | 0.0098
0.0916 | 0.1836 | -0.0920
0.1126 | 0.1231 | -0.0105
0.1846 | 0.1397 | 0.0450
0.1045 | 0.1244 | -0.0199
0.1404 | 0.1628 | -0.0224
0.2508 | 0.1188 | 0.1321

Removing alkphos_max from current feature set.
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
po2_median (loop 1 of 3, feature 2 of 84)
Total elapsed time: 0.08 minutes.
Loop elapsed time: 0.08 minutes.
Original mean logreg AUC: 0.1495
Temp mean logreg AUC: 0.1498
mean of diffs = 0.0003
sd of diffs = 0.0014
t = 0.

## Logreg only, 10 iter

In [27]:
trimmed_auprc_imp_v1_logreg_only_10_iter = iteratively_trim_feature_set(
    imp_v1_df,
    intermediate_feature_set_dict['auprc_v1'],
    'average_precision', # 'roc_auc' or 'average_precision'
    imp_method_str='med',
    # n_loops = 3,
    random_state_list=random_states_10_iter,
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=initial_logreg_scores_imp_v1_auprc,
    original_feature_set_svc_scores=initial_svc_scores_imp_v1_auprc,
    max_iter_svc=2500,
    allow_overwriting_files=True,
    logreg_only=True
)

with open('trimmed_feature_sets/auprc_med_10_iter_logreg_only.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auprc_imp_v1_logreg_only_10_iter, outfile)

NameError: name 'random_states_10_iter' is not defined

# Latest lab and median imputation, AUPRC

## Initial CV scores

In [17]:
initial_logreg_scores_imp_v2_auprc, initial_svc_scores_imp_v2_auprc = get_or_load_initial_scores(
    imp_v2_df,
    intermediate_feature_set_dict['auprc_v2'],
    'average_precision',
    logreg_score_file_path='intermediate_set_cv_scores/imp_v2_auprc_logreg.pickle',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    get_svc_scores=True
)

Calculating full logistic regression scores...
Calculating full SVC scores...


In [37]:
# initial_logreg_scores_imp_v2_auprc = train_logreg_cv_from_feature_set(imp_v2_df, 
#                                                                       intermediate_feature_set_dict['imp_v2_auprc'], 
#                                                                       scoring_metric='average_precision',
#                                                                       n_splits=10,
#                                                                       n_jobs=10,
#                                                                       random_state=343)

# print('Mean AUPRC for log reg trained on full intermediate feature set: %.3f' % np.mean(initial_logreg_scores_imp_v2_auprc))

Mean AUPRC for log reg trained on full intermediate feature set: 0.180


In [38]:
# initial_svc_scores_imp_v2_auprc = train_svc_cv_from_feature_set(imp_v2_df, 
#                                                               intermediate_feature_set_dict['imp_v2_auprc'], 
#                                                               scoring_metric='average_precision',
#                                                               n_splits=10,
#                                                               n_jobs=10,
#                                                               random_state=343,
#                                                                max_iter=2500)

# print('Mean AUPRC for SVC trained on full intermediate feature set: %.3f' % np.mean(initial_svc_scores_imp_v2_auprc))

Mean AUPRC for SVC trained on full intermediate feature set: 0.166


## Logreg and SVC

In [ ]:
trimmed_auprc_imp_v2 = iteratively_trim_feature_set(
    imp_v2_df,
    intermediate_feature_set_dict['auprc_v2'],
    'average_precision', # 'roc_auc' or 'average_precision'
    imp_method_str='ll',
    # n_loops = 3,
    random_state_list=[0, 42, 343],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=initial_logreg_scores_imp_v2_auprc,
    original_feature_set_svc_scores=initial_svc_scores_imp_v2_auprc,
    max_iter_svc=2500,
    allow_overwriting_files=False
)

with open('trimmed_feature_sets/auprc_ll_3_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auprc_imp_v2, outfile)

## Logreg only, 3 iter

In [39]:
if os.path.isfile('trimmed_feature_sets/auprc_ll_3_iter_logreg_only.pickle') == False:
    trimmed_auprc_imp_v2_logreg_only = iteratively_trim_feature_set(
        imp_v2_df,
        intermediate_feature_set_dict['auprc_v2'],
        'average_precision', # 'roc_auc' or 'average_precision'
        imp_method_str='ll',
        # n_loops = 3,
        random_state_list=[0, 42, 343],
        pval_threshold = 0.05,
        final_feature_set_list = [],
        random_state=343,
        original_feature_set_logreg_scores=initial_logreg_scores_imp_v2_auprc,
        original_feature_set_svc_scores=initial_svc_scores_imp_v2_auprc,
        max_iter_svc=2500,
        allow_overwriting_files=False,
        logreg_only=True
    )

with open('trimmed_feature_sets/auprc_ll_3_iter_logreg_only.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auprc_imp_v2_logreg_only, outfile)

Starting iteration.
Starting loop 1 after 0.00 minutes.
Starting loop 2 after 0.00 minutes.
Starting loop 3 after 0.00 minutes.


## Logreg only, 10 iter

In [40]:
if os.path.isfile('trimmed_feature_sets/auprc_ll_10_iter_logreg_only.pickle') == False:
    trimmed_auprc_imp_v2_logreg_only = iteratively_trim_feature_set(
        imp_v2_df,
        intermediate_feature_set_dict['auprc_v2'],
        'average_precision', # 'roc_auc' or 'average_precision'
        imp_method_str='ll',
        # n_loops = 3,
        random_state_list=random_states_10_iter,
        pval_threshold = 0.05,
        final_feature_set_list = [],
        random_state=343,
        original_feature_set_logreg_scores=initial_logreg_scores_imp_v2_auprc,
        original_feature_set_svc_scores=initial_svc_scores_imp_v2_auprc,
        max_iter_svc=2500,
        allow_overwriting_files=False,
        logreg_only=True
    )

with open('trimmed_feature_sets/auprc_ll_10_iter_logreg_only.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auprc_imp_v2_logreg_only, outfile)

# Median-only imputation v1, AUROC

## Initial CV scores

In [30]:
initial_logreg_scores_imp_v1_auroc, initial_svc_scores_imp_v1_auroc = get_or_load_initial_scores(
    imp_v1_df,
    intermediate_feature_set_dict['auroc_v1'],
    'roc_auc',
    logreg_score_file_path='intermediate_set_cv_scores/imp_v1_auroc_logreg.pickle',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    get_svc_scores=True
)

Loading full logistic regression scores...
Calculating full SVC scores...


## Logreg and SVC

In [36]:
trimmed_auroc_imp_v1 = iteratively_trim_feature_set(
    imp_v1_df,
    intermediate_feature_set_dict['auroc_v1'],
    'roc_auc', # 'roc_auc' or 'average_precision'
    imp_method_str='med',
    # n_loops = 3,
    random_state_list=[0, 42, 343],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=initial_logreg_scores_imp_v1_auroc,
    original_feature_set_svc_scores=initial_svc_scores_imp_v1_auroc,
    max_iter_svc=2500,
    allow_overwriting_files=True
)

with open('trimmed_feature_sets/auroc_med_3_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_imp_v1, outfile)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
hr_median (loop 1 of 3, feature 1 of 83)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.7749
Temp mean logreg AUC: 0.7739
mean of diffs = -0.0010
sd of diffs = 0.0530
t = -0.0612
Logistic regression model not significantly worse after removing hr_median.
Logistic regression scores (new | old | new - old):
0.8025 | 0.7242 | 0.0783
0.7792 | 0.7568 | 0.0224
0.8135 | 0.7756 | 0.0379
0.7244 | 0.8177 | -0.0934
0.7336 | 0.8032 | -0.0696
0.7500 | 0.7496 | 0.0004
0.8214 | 0.7607 | 0.0607
0.7382 | 0.7987 | -0.0604
0.7660 | 0.7897 | -0.0237
0.8102 | 0.7731 | 0.0371

Assessing SVC performance...
Original mean SVC AUC: 0.3365
Temp mean SVC AUC: 0.3387
mean of diffs = 0.0022
sd of diffs = 0.0521
t = 0.1310
SVC model also not significantly worse after removing hr_median.
SVC scores (new | old | new - old):
0.3375 | 0.3589 | -0.0213
0.3557 | 0.2575 | 0.0981
0.3032 | 0.2953 | 0.0

## Logreg only, 3 iter

In [31]:
trimmed_auroc_imp_v1_logreg_only = iteratively_trim_feature_set(
    imp_v1_df,
    intermediate_feature_set_dict['auroc_v1'],
    'roc_auc', # 'roc_auc' or 'average_precision'
    imp_method_str='med',
    # n_loops = 3,
    random_state_list=[0, 42, 343],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=initial_logreg_scores_imp_v1_auroc,
    original_feature_set_svc_scores=initial_svc_scores_imp_v1_auroc,
    max_iter_svc=2500,
    allow_overwriting_files=True,
    logreg_only=True
)

with open('trimmed_feature_sets/auroc_med_3_iter_logreg_only.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_imp_v1_logreg_only, outfile)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
bicarb_max (loop 1 of 3, feature 1 of 83)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.7749
Temp mean logreg AUC: 0.7745
mean of diffs = -0.0005
sd of diffs = 0.0537
t = -0.0277
Logistic regression model not significantly worse after removing bicarb_max.
Logistic regression scores (new | old | new - old):
0.8052 | 0.7242 | 0.0810
0.7780 | 0.7568 | 0.0213
0.8145 | 0.7756 | 0.0389
0.7240 | 0.8177 | -0.0937
0.7314 | 0.8032 | -0.0718
0.7530 | 0.7496 | 0.0035
0.8215 | 0.7607 | 0.0608
0.7387 | 0.7987 | -0.0600
0.7666 | 0.7897 | -0.0231
0.8116 | 0.7731 | 0.0385

Removing bicarb_max from current feature set.
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
be_min (loop 1 of 3, feature 2 of 83)
Total elapsed time: 0.05 minutes.
Loop elapsed time: 0.05 minutes.
Original mean logreg AUC: 0.7749
Temp mean logreg AUC: 0.7746
mean of diffs = -0.0003
sd of diffs = 0.0541
t = -0.0171

## Logreg only, 10 iter

In [32]:
trimmed_auroc_imp_v1_logreg_only_10_iter = iteratively_trim_feature_set(
    imp_v1_df,
    intermediate_feature_set_dict['auroc_v1'],
    'roc_auc', # 'roc_auc' or 'average_precision'
    imp_method_str='med',
    # n_loops = 3,
    random_state_list=random_states_10_iter,
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=initial_logreg_scores_imp_v1_auroc,
    original_feature_set_svc_scores=initial_svc_scores_imp_v1_auroc,
    max_iter_svc=2500,
    allow_overwriting_files=True,
    logreg_only=True
)

with open('trimmed_feature_sets/auroc_med_10_iter_logreg_only.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_imp_v1_logreg_only_10_iter, outfile)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
cal_median (loop 1 of 10, feature 1 of 83)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.7749
Temp mean logreg AUC: 0.7684
mean of diffs = -0.0065
sd of diffs = 0.0307
t = -0.6736
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
cl_max (loop 1 of 10, feature 2 of 83)
Total elapsed time: 0.06 minutes.
Loop elapsed time: 0.06 minutes.
Original mean logreg AUC: 0.7749
Temp mean logreg AUC: 0.7694
mean of diffs = -0.0056
sd of diffs = 0.0287
t = -0.6132
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
dbp_mean (loop 1 of 10, feature 3 of 83)
Total elapsed time: 0.11 minutes.
Loop elapsed time: 0.11 minutes.
Original mean logreg AUC: 0.7749
Temp mean logreg AUC: 0.7703
mean of diffs = -0.0047
sd of diffs = 0.0285
t = -0.5184
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
na_median (loop 1 of 10, feature 4 of 83)
Total elapsed time: 0.17 minutes.
Loop elapsed time: 0.17 minutes.
Origina

In [ ]:
# final_feature_set_dict['imp_v1_auroc'] = iteratively_trim_feature_set(imp_v1_df,
#                                                                      intermediate_feature_set_dict['imp_v1_auroc'],
#                                                                      'roc_auc', # 'roc_auc_score' or 'average_precision'
#                                                                      n_loops = 3,
#                                                                      pval_threshold = 0.01,
#                                                                      final_feature_set_list = [],
#                                                                      random_state=343,
#                                                                      full_feature_set_logreg_scores=initial_logreg_scores_imp_v1_auroc,
#                                                                      full_feature_set_svc_scores=initial_svc_scores_imp_v1_auroc)

In [ ]:
# for i in range(len(final_feature_set_dict['imp_v1_auroc'])):
#     print(len(final_feature_set_dict['imp_v1_auroc'][i]))

# Latest lab and median imputation v2, AUROC

## Initial CV scores

In [33]:
initial_logreg_scores_imp_v2_auroc, initial_svc_scores_imp_v2_auroc = get_or_load_initial_scores(
    imp_v2_df,
    intermediate_feature_set_dict['auroc_v2'],
    'roc_auc',
    logreg_score_file_path='intermediate_set_cv_scores/imp_v2_auroc_logreg.pickle',
    n_splits=10,
    n_jobs=10,
    random_state=343,
    max_iter_svc=2500,
    get_svc_scores=True
)

Calculating full logistic regression scores...
Calculating full SVC scores...


In [ ]:
# initial_logreg_scores_imp_v2_auroc = train_logreg_cv_from_feature_set(imp_v2_df, 
#                                                                       intermediate_feature_set_dict['imp_v2_auroc'], 
#                                                                       scoring_metric='roc_auc',
#                                                                       n_splits=10,
#                                                                       n_jobs=10,
#                                                                       random_state=343)

# print('Mean AUROC for log reg trained on full intermediate feature set: %.3f' % np.mean(initial_logreg_scores_imp_v2_auroc))

In [ ]:
# initial_svc_scores_imp_v2_auroc = train_svc_cv_from_feature_set(imp_v2_df, 
#                                                               intermediate_feature_set_dict['imp_v2_auroc'], 
#                                                               scoring_metric='roc_auc',
#                                                               n_splits=10,
#                                                               n_jobs=10,
#                                                               random_state=343,
#                                                               max_iter=2500)

# print('Mean AUROC for SVC trained on full intermediate feature set: %.3f' % np.mean(initial_svc_scores_imp_v2_auroc))

## Logreg and SVC

In [37]:
trimmed_auroc_imp_v2 = iteratively_trim_feature_set(
    imp_v2_df,
    intermediate_feature_set_dict['auroc_v2'],
    'roc_auc', # 'roc_auc' or 'average_precision'
    imp_method_str='ll',
    # n_loops = 3,
    random_state_list=[0, 42, 343],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=initial_logreg_scores_imp_v2_auroc,
    original_feature_set_svc_scores=initial_svc_scores_imp_v2_auroc,
    max_iter_svc=2500,
    allow_overwriting_files=True
)

with open('trimmed_feature_sets/auroc_ll_3_iter.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_imp_v2, outfile)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
dbp_mean (loop 1 of 3, feature 1 of 83)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.7968
Temp mean logreg AUC: 0.7961
mean of diffs = -0.0007
sd of diffs = 0.0456
t = -0.0480
Logistic regression model not significantly worse after removing dbp_mean.
Logistic regression scores (new | old | new - old):
0.8207 | 0.7562 | 0.0645
0.8108 | 0.7799 | 0.0308
0.8462 | 0.8172 | 0.0290
0.7583 | 0.8369 | -0.0786
0.7732 | 0.8225 | -0.0492
0.7576 | 0.7790 | -0.0214
0.8381 | 0.7799 | 0.0581
0.7651 | 0.8140 | -0.0489
0.7774 | 0.8053 | -0.0280
0.8142 | 0.7774 | 0.0368

Assessing SVC performance...
Original mean SVC AUC: 0.3552
Temp mean SVC AUC: 0.3610
mean of diffs = 0.0059
sd of diffs = 0.0442
t = 0.4197
SVC model also not significantly worse after removing dbp_mean.
SVC scores (new | old | new - old):
0.3443 | 0.3935 | -0.0492
0.3671 | 0.3052 | 0.0620
0.3113 | 0.3402 | -0.02

## Logreg only, 3 iter

In [34]:
trimmed_auroc_imp_v2_logreg_only = iteratively_trim_feature_set(
    imp_v2_df,
    intermediate_feature_set_dict['auroc_v2'],
    'roc_auc', # 'roc_auc' or 'average_precision'
    imp_method_str='ll',
    # n_loops = 3,
    random_state_list=[0, 42, 343],
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=initial_logreg_scores_imp_v2_auroc,
    original_feature_set_svc_scores=initial_svc_scores_imp_v2_auroc,
    max_iter_svc=2500,
    allow_overwriting_files=True,
    logreg_only=True
)

with open('trimmed_feature_sets/auroc_ll_3_iter_logreg_only.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_imp_v2_logreg_only, outfile)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
uprot_max (loop 1 of 3, feature 1 of 83)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.7968
Temp mean logreg AUC: 0.7962
mean of diffs = -0.0006
sd of diffs = 0.0453
t = -0.0417
Logistic regression model not significantly worse after removing uprot_max.
Logistic regression scores (new | old | new - old):
0.8216 | 0.7562 | 0.0654
0.8092 | 0.7799 | 0.0293
0.8449 | 0.8172 | 0.0277
0.7577 | 0.8369 | -0.0792
0.7754 | 0.8225 | -0.0470
0.7576 | 0.7790 | -0.0214
0.8375 | 0.7799 | 0.0575
0.7668 | 0.8140 | -0.0472
0.7772 | 0.8053 | -0.0281
0.8146 | 0.7774 | 0.0371

Removing uprot_max from current feature set.
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
be_min (loop 1 of 3, feature 2 of 83)
Total elapsed time: 0.06 minutes.
Loop elapsed time: 0.06 minutes.
Original mean logreg AUC: 0.7968
Temp mean logreg AUC: 0.7963
mean of diffs = -0.0005
sd of diffs = 0.0459
t = -0.0338
L

## Logreg only, 10 iter

In [35]:
trimmed_auroc_imp_v2_logreg_only_10_iter = iteratively_trim_feature_set(
    imp_v2_df,
    intermediate_feature_set_dict['auroc_v2'],
    'roc_auc', # 'roc_auc' or 'average_precision'
    imp_method_str='ll',
    # n_loops = 3,
    random_state_list=random_states_10_iter,
    pval_threshold = 0.05,
    final_feature_set_list = [],
    random_state=343,
    original_feature_set_logreg_scores=initial_logreg_scores_imp_v2_auroc,
    original_feature_set_svc_scores=initial_svc_scores_imp_v2_auroc,
    max_iter_svc=2500,
    allow_overwriting_files=True,
    logreg_only=True
)

with open('trimmed_feature_sets/auroc_ll_10_iter_logreg_only.pickle', 'wb') as outfile:
    pickle.dump(trimmed_auroc_imp_v2_logreg_only_10_iter, outfile)

Starting iteration.
Starting loop 1 after 0.00 minutes.
~~~~~~~~~~~~~~~~~~~~
cl_median (loop 1 of 10, feature 1 of 83)
Total elapsed time: 0.00 minutes.
Loop elapsed time: 0.00 minutes.
Original mean logreg AUC: 0.7968
Temp mean logreg AUC: 0.7927
mean of diffs = -0.0041
sd of diffs = 0.0267
t = -0.4880
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
Total Bilirubin_mean (loop 1 of 10, feature 2 of 83)
Total elapsed time: 0.06 minutes.
Loop elapsed time: 0.06 minutes.
Original mean logreg AUC: 0.7968
Temp mean logreg AUC: 0.7933
mean of diffs = -0.0035
sd of diffs = 0.0277
t = -0.3991
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
cal_median (loop 1 of 10, feature 3 of 83)
Total elapsed time: 0.12 minutes.
Loop elapsed time: 0.12 minutes.
Original mean logreg AUC: 0.7968
Temp mean logreg AUC: 0.7921
mean of diffs = -0.0047
sd of diffs = 0.0303
t = -0.4900
~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~
Platelet Count_max (loop 1 of 10, feature 4 of 83)
Total elapsed time: 0.18 minutes.
Loop elapsed tim

In [ ]:
# final_feature_set_dict['imp_v2_auroc'] = iteratively_trim_feature_set(imp_v2_df,
#                                                                      intermediate_feature_set_dict['imp_v2_auroc'],
#                                                                      'roc_auc', # 'roc_auc_score' or 'average_precision'
#                                                                      n_loops = 3,
#                                                                      pval_threshold = 0.01,
#                                                                      final_feature_set_list = [],
#                                                                      random_state=343,
#                                                                      full_feature_set_logreg_scores=initial_logreg_scores_imp_v2_auroc,
#                                                                      full_feature_set_svc_scores=initial_svc_scores_imp_v2_auroc,
#                                                                      max_iter_svc=2500)

In [51]:
for i in range(len(final_feature_set_dict['imp_v2_auroc'])):
    print(len(final_feature_set_dict['imp_v2_auroc'][i]))

11
11
11


# Save dictionary of feature sets

In [ ]:
with open('pickle/2024-02-09 - Final Feature Set Dictionary (Includes baseline_bSCr).pickle', 'wb') as outfile:
    pickle.dump(final_feature_set_dict, outfile)